# Backtest: Backpack Exchange Bar Data

Tutorial for [NautilusTrader](https://nautilustrader.io/docs/) a high-performance algorithmic trading platform and event driven backtester.

[View source on GitHub](https://github.com/nautechsystems/nautilus_trader/blob/develop/docs/tutorials/backtest_backpack_bars.ipynb).

## Overview

This tutorial demonstrates how to:
1. Fetch historical bar data from Backpack Exchange
2. Set up a data catalog for backtesting
3. Configure and run a backtest with an EMA crossover strategy
4. Analyze the backtest results

Backpack Exchange is a cryptocurrency exchange that provides public market data through their API. We'll use their klines (candlestick) endpoint to fetch historical OHLCV data.

## Prerequisites

- Python 3.11+ installed
- [JupyterLab](https://jupyter.org/) or similar installed (`pip install -U jupyterlab`)
- [NautilusTrader](https://pypi.org/project/nautilus_trader/) latest release installed (`pip install -U nautilus_trader`)
- aiohttp library for API requests (`pip install aiohttp`)

## Imports

We'll start with all of our imports for the remainder of this guide:

In [ ]:
import asyncio
import os
import shutil
from datetime import datetime, timedelta, timezone
from decimal import Decimal
from pathlib import Path

import pandas as pd
import aiohttp

from nautilus_trader.adapters.backpack.common.constants import BACKPACK_VENUE
from nautilus_trader.backtest.node import BacktestDataConfig
from nautilus_trader.backtest.node import BacktestEngineConfig
from nautilus_trader.backtest.node import BacktestNode
from nautilus_trader.backtest.node import BacktestRunConfig
from nautilus_trader.backtest.node import BacktestVenueConfig
from nautilus_trader.config import ImportableStrategyConfig
from nautilus_trader.config import LoggingConfig
from nautilus_trader.core.datetime import dt_to_unix_nanos
from nautilus_trader.model.currencies import Currency
from nautilus_trader.model.data import Bar
from nautilus_trader.model.data import BarType
from nautilus_trader.model.data import BarSpecification
from nautilus_trader.model.enums import AccountType
from nautilus_trader.model.enums import AggregationSource
from nautilus_trader.model.enums import BarAggregation
from nautilus_trader.model.enums import OmsType
from nautilus_trader.model.enums import PriceType
from nautilus_trader.model.identifiers import InstrumentId
from nautilus_trader.model.identifiers import Symbol
from nautilus_trader.model.instruments import CurrencyPair
from nautilus_trader.model.objects import Money
from nautilus_trader.model.objects import Price
from nautilus_trader.model.objects import Quantity
from nautilus_trader.persistence.catalog import ParquetDataCatalog
from nautilus_trader.model import Venue

## Fetching Historical Data from Backpack

First, we'll create a function to fetch historical klines (bar) data from Backpack Exchange's public API. The API doesn't require authentication for public market data.

In [ ]:
async def fetch_backpack_bars(
    symbol: str = "BTC_USDC",
    interval: str = "1h",
    days_back: int = 30,
) -> list:
    """
    Fetch historical klines/bars from Backpack Exchange.
    
    Parameters
    ----------
    symbol : str
        The trading pair symbol (e.g., 'BTC_USDC')
    interval : str
        The bar interval (1m, 5m, 15m, 1h, 4h, 1d)
    days_back : int
        Number of days of historical data to fetch
    
    Returns
    -------
    list
        Raw klines data from Backpack API
    
    """
    # Calculate time range
    end_time = datetime.now(timezone.utc)
    start_time = end_time - timedelta(days=days_back)
    
    # Convert to seconds (Backpack API uses seconds, not milliseconds)
    start_timestamp = int(start_time.timestamp())
    end_timestamp = int(end_time.timestamp())
    
    print(f"Fetching {symbol} {interval} bars from {start_time} to {end_time}")
    
    # Fetch klines directly from the public API
    url = "https://api.backpack.exchange/api/v1/klines"
    params = {
        "symbol": symbol,
        "interval": interval,
        "startTime": start_timestamp,
        "endTime": end_timestamp,
    }
    
    async with aiohttp.ClientSession() as session:
        async with session.get(url, params=params) as response:
            if response.status == 200:
                klines = await response.json()
                print(f"Fetched {len(klines)} bars from Backpack")
                return klines
            else:
                error_text = await response.text()
                raise Exception(f"Failed to fetch klines: {response.status} - {error_text}")

## Creating a Backpack Instrument

Next, we need to create an instrument definition that matches the Backpack trading pair. We'll detect the price and size precision from the actual data to ensure compatibility.

In [ ]:
def create_backpack_instrument(
    symbol: str = "BTC_USDC", 
    price_precision: int = 1,
    size_precision: int = 8,
) -> CurrencyPair:
    """
    Create a Backpack instrument for backtesting.
    
    Parameters
    ----------
    symbol : str
        The trading pair symbol
    price_precision : int
        The price precision (will be auto-detected from data)
    size_precision : int
        The size/volume precision (will be auto-detected from data)
    
    Returns
    -------
    CurrencyPair
        The configured instrument
    
    """
    base, quote = symbol.split("_")
    
    # Set price increment based on precision
    price_increment_str = "0." + "0" * (price_precision - 1) + "1" if price_precision > 0 else "1"
    size_increment_str = "0." + "0" * (size_precision - 1) + "1" if size_precision > 0 else "1"
    
    return CurrencyPair(
        instrument_id=InstrumentId(
            symbol=Symbol(symbol),
            venue=BACKPACK_VENUE,
        ),
        raw_symbol=Symbol(symbol),
        base_currency=Currency.from_str(base),
        quote_currency=Currency.from_str(quote),
        price_precision=price_precision,
        size_precision=size_precision,
        price_increment=Price.from_str(price_increment_str),
        size_increment=Quantity.from_str(size_increment_str),
        lot_size=None,
        max_quantity=Quantity.from_str("10000"),
        min_quantity=Quantity.from_str("0.0001"),
        max_notional=None,
        min_notional=Money(10, Currency.from_str(quote)),  # $10 minimum
        max_price=Price.from_str("10000000"),
        min_price=Price.from_str("0.01"),
        margin_init=Decimal("0"),
        margin_maint=Decimal("0"),
        maker_fee=Decimal("0.0002"),  # 0.02%
        taker_fee=Decimal("0.0005"),  # 0.05%
        ts_event=0,
        ts_init=0,
    )

## Processing Klines Data

Now we'll convert the raw klines data from Backpack into NautilusTrader Bar objects:

In [ ]:
def process_klines_to_bars(
    klines: list,
    instrument: CurrencyPair,
    bar_type: BarType,
) -> list[Bar]:
    """
    Process raw klines data into NautilusTrader Bar objects.
    
    Parameters
    ----------
    klines : list
        Raw klines data from Backpack (can be list of lists or list of dicts)
    instrument : CurrencyPair
        The instrument for the bars
    bar_type : BarType
        The bar type specification
    
    Returns
    -------
    list[Bar]
        Processed bar objects
    
    """
    bars = []
    
    for kline in klines:
        # Check if kline is a dict or list
        if isinstance(kline, dict):
            # Dictionary format from API
            timestamp_ms = kline.get('start', kline.get('timestamp', 0))
            open_price = kline.get('open', 0)
            high_price = kline.get('high', 0)
            low_price = kline.get('low', 0)
            close_price = kline.get('close', 0)
            volume = kline.get('volume', 0)
        else:
            # List format: [timestamp, open, high, low, close, volume, ...]
            timestamp_ms = kline[0]
            open_price = kline[1]
            high_price = kline[2]
            low_price = kline[3]
            close_price = kline[4]
            volume = kline[5]
        
        # Convert timestamp to nanoseconds
        # Handle both string and numeric timestamps
        if isinstance(timestamp_ms, str):
            # Parse string timestamp
            from dateutil import parser
            dt = parser.parse(timestamp_ms)
            timestamp_ms = int(dt.timestamp() * 1000)
        
        ts_event = int(timestamp_ms) * 1_000_000
        ts_init = ts_event
        
        bar = Bar(
            bar_type=bar_type,
            open=Price.from_str(str(open_price)),
            high=Price.from_str(str(high_price)),
            low=Price.from_str(str(low_price)),
            close=Price.from_str(str(close_price)),
            volume=Quantity.from_str(str(volume)),
            ts_event=ts_event,
            ts_init=ts_init,
        )
        bars.append(bar)
    
    return bars

## Main Backtest Workflow

Now let's put it all together and run a complete backtest:

In [ ]:
# Configuration
symbol = "BTC_USDC"
interval = "1h"
days_back = 30

# Fetch historical data from Backpack
print("Fetching historical data from Backpack Exchange...")
klines = await fetch_backpack_bars(
    symbol=symbol,
    interval=interval,
    days_back=days_back,
)

if not klines:
    print("No data fetched. Please check your connection and try again.")
else:
    print(f"Successfully fetched {len(klines)} bars")

In [ ]:
# Detect price and volume precision from the first kline
if klines and len(klines) > 0:
    first_kline = klines[0]
    if isinstance(first_kline, dict):
        sample_price = str(first_kline.get('open', 0))
        sample_volume = str(first_kline.get('volume', 0))
    else:
        sample_price = str(first_kline[1])
        sample_volume = str(first_kline[5])
    
    # Determine price precision from decimal places
    if '.' in sample_price:
        decimal_places = len(sample_price.split('.')[1].rstrip('0'))
        price_precision = max(1, decimal_places)  # At least 1 decimal place
    else:
        price_precision = 0
        
    # Determine volume precision from decimal places
    if '.' in sample_volume:
        decimal_places = len(sample_volume.split('.')[1].rstrip('0'))
        size_precision = max(1, decimal_places)  # At least 1 decimal place
    else:
        size_precision = 0
else:
    price_precision = 1  # Default
    size_precision = 3  # Default

print(f"Detected price precision: {price_precision}, size precision: {size_precision}")

# Create instrument
instrument = create_backpack_instrument(symbol, price_precision, size_precision)
print(f"Created instrument: {instrument.id}")

In [ ]:
# Create bar type
bar_type = BarType(
    instrument_id=instrument.id,
    bar_spec=BarSpecification(
        step=1,
        aggregation=BarAggregation.HOUR,
        price_type=PriceType.LAST,
    ),
    aggregation_source=AggregationSource.EXTERNAL,
)

# Process klines to bars
bars = process_klines_to_bars(klines, instrument, bar_type)
print(f"Processed {len(bars)} bars")

# Display a sample of the data
if bars:
    print(f"\nFirst bar: {bars[0]}")
    print(f"Last bar: {bars[-1]}")

## Setting Up the Data Catalog

We'll store the processed data in a ParquetDataCatalog for efficient access during backtesting:

In [ ]:
# Set up data catalog
CATALOG_PATH = os.getcwd() + "/backpack_catalog"

# Clear if it already exists, then create fresh
if os.path.exists(CATALOG_PATH):
    shutil.rmtree(CATALOG_PATH)
os.mkdir(CATALOG_PATH)

# Create a catalog instance
catalog = ParquetDataCatalog(CATALOG_PATH)

# Write instrument and bars to catalog
catalog.write_data([instrument])
catalog.write_data(bars)

print(f"Data catalog created at: {CATALOG_PATH}")
print(f"Instruments in catalog: {catalog.instruments()}")

## Configuring the Backtest

Now we'll configure the backtest with:
- Venue configuration for Backpack Exchange
- Data configuration pointing to our catalog
- Strategy configuration (EMA crossover strategy)

In [ ]:
# Configure data
data_configs = [
    BacktestDataConfig(
        catalog_path=CATALOG_PATH,
        data_cls=Bar,
        instrument_id=instrument.id,
        bar_spec=bar_type.spec,  # Important: specify the bar_spec
    )
]

# Configure venue
venues_configs = [
    BacktestVenueConfig(
        name="BACKPACK",
        oms_type="NETTING",  # Single position per instrument
        account_type="CASH",  # Spot trading
        base_currency=None,
        starting_balances=["10000 USDC", "0.1 BTC"],  # Initial capital
    )
]

# Configure strategy
strategies = [
    ImportableStrategyConfig(
        strategy_path="nautilus_trader.examples.strategies.ema_cross:EMACross",
        config_path="nautilus_trader.examples.strategies.ema_cross:EMACrossConfig",
        config={
            "instrument_id": str(instrument.id),
            "bar_type": str(bar_type),
            "fast_ema_period": 10,
            "slow_ema_period": 20,
            "trade_size": Decimal("0.001"),  # Trade 0.001 BTC per signal
        },
    ),
]

# Create run configuration
config = BacktestRunConfig(
    engine=BacktestEngineConfig(
        strategies=strategies,
        logging=LoggingConfig(log_level="INFO"),
    ),
    data=data_configs,
    venues=venues_configs,
)

print("Backtest configuration ready")

## Running the Backtest

In [ ]:
# Run backtest
node = BacktestNode(configs=[config])
result = node.run()

print(f"\nBacktest completed!")
if result:
    print(f"Run ID: {result[0].run_id}")

## Analyzing Results

Let's generate reports to analyze the backtest performance:

In [ ]:
from nautilus_trader.backtest.engine import BacktestEngine

# Get engine for detailed reports
engine: BacktestEngine = node.get_engine(config.id)

# Generate order fills report
fills_report = engine.trader.generate_order_fills_report()
if not fills_report.empty:
    print("\n=== Order Fills Report ===")
    print(fills_report.head(10))
else:
    print("\nNo trades executed")

In [ ]:
# Generate positions report
positions_report = engine.trader.generate_positions_report()
if not positions_report.empty:
    print("\n=== Positions Report ===")
    print(positions_report.head(10))
else:
    print("\nNo positions opened")

In [ ]:
# Generate account report
account_report = engine.trader.generate_account_report(BACKPACK_VENUE)
print("\n=== Account Report ===")
print(account_report)

## Summary

In this tutorial, we demonstrated how to:

1. **Fetch historical data** from Backpack Exchange using their public API
2. **Process the data** into NautilusTrader format with proper precision detection
3. **Set up a data catalog** for efficient data storage and retrieval
4. **Configure a backtest** with venue settings specific to Backpack Exchange
5. **Run a strategy** using the EMA crossover example
6. **Analyze results** using built-in reporting tools

### Key Takeaways

- Backpack Exchange provides free historical data through their public API
- Price and volume precision should be detected from actual data to avoid mismatches
- The data catalog allows efficient storage and reuse of historical data
- NautilusTrader provides comprehensive reporting for backtest analysis

### Next Steps

- Try different trading strategies from the examples
- Experiment with different timeframes and instruments
- Implement your own custom trading strategy
- Add more sophisticated risk management rules
- Explore live trading with the Backpack adapter